<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "729"
iso_country = "Sudan"
year = 2018
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 33.590*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Sudan': 

Number of trade records present in dataset: 11026
Number of partner countries present in dataset: 131
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$7597.7m
Total weight within the dataset: 11698.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,729,Sudan,682,Saudi Arabia,Import,Sea,10121,"010121 | Horses; live, pure-b",0101,"0101 | Horses, asses, mules",01,01 | Animals; live,NaN,NaN,NaN,NaN,NaN,8160,NaN,8160,NaN,0.000,11000.0,556.512,232.0,144.158072,8.022570e+01,['Container'],['Container'],0.295609,0.294740,0.294814,0.294814,0.295310,0.295310,0.310471,0.573456,0.342506,0.316267,0.317390,0.326045,0.303497,0.553656,0.359069,0.328754,0.320846,0.336870,0.297375,0.607643,0.445617,0.390350,0.360303,0.351847,0.291358,0.642802,0.521087,0.485348,0.390135,0.378344,0.285458,0.666504,0.629812,0.638201,0.415949,0.398022,0.279444,0.647285,0.629822,0.673424,0.511089,0.405530,0.273507,0.572701,0.586610,0.630482,0.472630,0.410417,0.267749,0.515075,0.528534,0.632181,0.514320,0.413509,nan
1,1,729,Sudan,528,Netherlands,Import,Sea,10229,"010229 | Cattle; live, other",0102,0102 | Bovine animals; live,01,01 | Animals; live,NaN,NaN,NaN,NaN,NaN,5774000,NaN,5774000,u,5485.445,794542.0,1900223.400,7643.0,4749.138553,9.024424e+06,['General cargo'],['General cargo'],61415.567861,61380.203669,61381.710497,61381.710497,61398.424957,61398.424957,63088.711763,93165.441966,65723.672209,63047.811443,62989.965231,64889.722578,62297.267845,88802.281397,67495.752016,65372.012813,64526.039840,66118.683634,61603.132566,97177.675103,78727.937755,73888.005389,70493.303239,67805.009705,60921.857274,106299.104139,87268.939520,83873.898218,72559.618182,70825.825401,60253.441746,109894.158858,100107.742560,108328.588570,76309.945177,73033.977057,59571.627699,101362.855190,96913.754800,106816.013453,89499.691715,73856.973599,58929.310153,96096.728858,103918.096095,107679.909091,89594.575462,74398.947371,58274.350551,93277.433351,94653.749351,106704.191019,90236.048495,74750.842553,nan


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $2872.7m (37.8%), ... 

                             cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               1745818397  6.274944e+09     23.0     53.6
17 | Sugars and sugar con   631536066  1.372032e+09      8.3     11.7
15 | Animal, vegetable or   115621098  9.578506e+07      1.5      0.8
09 | Coffee, tea, mate an    97051787  2.522225e+07      1.3      0.2
07 | Vegetables and certa    94137395  1.411903e+08      1.2      1.2
21 | Miscellaneous edible    64546852  2.504072e+07      0.8      0.2
04 | Dairy produce; birds    47371597  1.695316e+07      0.6      0.1
20 | Preparations of vege    25058752  2.087411e+07      0.3      0.2
23 | Food industries, res    22387070  1.656476e+07      0.3      0.1
11 | Products of the mill     6970776  1.172196e+07      0.1      0.1
12 | Oil seeds and oleagi     6813730  5.355372e+06      0

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1148.6m (15.1%), ... 

                   cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Food & Beverages  746797168  1.435177e+09      9.8     12.3
Agriculture       401739293  3.520442e+08      5.3      3.0
Fishing              100106  2.025170e+04      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1148.2m (15.1%), ... 

                                cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
SGR - Sugar                    616920465  1.353410e+09      8.1     11.6
OFD - Food products n.e.c.     137633121  9.342368e+07      1.8      0.8
VOL - Vegetable oils and fats  105404310  8.620679e+07      1.4      0.7
OCR - Crops n.e.c.              98501447  2.530651e+07      1.3      0.2
V_F - Vegetables, fruit, nuts   93984612  1.405542e+08      1.2      1.2
MIL - Dairy products            47204817  1.692060e+07      0.6      0.1
PDR - Paddy rice                30992353  5.383372e+07      0.4      0.5
OMT - Meat products n.e.c.      10362294  9.617652e+06      0.1      0.1
OSD - Oil seeds                  3738699  4.738452e+06      0.0      0.0
CMT - Bovine meat prods          2474830  5.036094e+05      0.0      0.0
GRO - Cereal grains n.e.c.        414179  1.712396e+06   

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 37.8% 
GTAP 15.1% 
EORA 15.1%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46425/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,729,Sudan,682,Saudi Arabia,Import,Sea,10121,"010121 | Horses; live, pure-b",0101,"0101 | Horses, asses, mules",01,01 | Animals; live,NaN,NaN,NaN,NaN,NaN,8160,NaN,8160,NaN,0.000,11000.0,556.512,232.0,144.158072,8.022570e+01,['Container'],['Container'],0.295609,0.294740,0.294814,0.294814,0.295310,0.295310,0.310471,0.573456,0.342506,0.316267,0.317390,0.326045,0.303497,0.553656,0.359069,0.328754,0.320846,0.336870,0.297375,0.607643,0.445617,0.390350,0.360303,0.351847,0.291358,0.642802,0.521087,0.485348,0.390135,0.378344,0.285458,0.666504,0.629812,0.638201,0.415949,0.398022,0.279444,0.647285,0.629822,0.673424,0.511089,0.405530,0.273507,0.572701,0.586610,0.630482,0.472630,0.410417,0.267749,0.515075,0.528534,0.632181,0.514320,0.413509,nan,NaN
1,1,729,Sudan,528,Netherlands,Import,Sea,10229,"010229 | Cattle; live, other",0102,0102 | Bovine animals; live,01,01 | Animals; live,NaN,NaN,NaN,NaN,NaN,5774000,NaN,5774000,u,5485.445,794542.0,1900223.400,7643.0,4749.138553,9.024424e+06,['General cargo'],['General cargo'],61415.567861,61380.203669,61381.710497,61381.710497,61398.424957,61398.424957,63088.711763,93165.441966,65723.672209,63047.811443,62989.965231,64889.722578,62297.267845,88802.281397,67495.752016,65372.012813,64526.039840,66118.683634,61603.132566,97177.675103,78727.937755,73888.005389,70493.303239,67805.009705,60921.857274,106299.104139,87268.939520,83873.898218,72559.618182,70825.825401,60253.441746,109894.158858,100107.742560,108328.588570,76309.945177,73033.977057,59571.627699,101362.855190,96913.754800,106816.013453,89499.691715,73856.973599,58929.310153,96096.728858,103918.096095,107679.909091,89594.575462,74398.947371,58274.350551,93277.433351,94653.749351,106704.191019,90236.048495,74750.842553,nan,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $2674.6m of $7597.7m (35.2%)
Volume: 7918.3 of 11698.4 million kilograms (67.7%)
Value-to-Weight: 0.34 from 0.65 US$ per kg (+-48.0%)
Average distance to market (volume-weighted): 2729.1 from 2677.7 nm (+1.9%)

            food_group  N_com    cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     22  1714621969  6.220756e+09     22.6     53.2
1      Sugar and Honey     50   631536066  1.372032e+09      8.3     11.7
2        Fats and Oils     46   113413760  9.325504e+07      1.5      0.8
3           Vegetables     65    97034063  1.423373e+08      1.3      1.2
4    Processed Cereals    154    67773894  7.163362e+07      0.9      0.6
5                Dairy     56    47178032  1.689501e+07      0.6      0.1
6        Meat and Prep     23     2400142  8.301596e+05      0.0      0.0
7               Fruits     38      676455  5.151359e+05      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,5.299637e+07,9.784353e+07,5.845677e+07,5.398235e+07,5.417405e+07,5.565231e+07,4.873043e+07,1.137444e+08,1.074732e+08,1.089112e+08,7.098534e+07,6.792869e+07,4.571004e+07,8.790047e+07,9.019026e+07,1.078620e+08,8.778095e+07,7.057056e+07
1,Fats and Oils,1.403755e+08,2.416198e+08,1.520101e+08,1.415366e+08,1.419795e+08,1.464233e+08,1.306596e+08,2.767607e+08,2.659369e+08,2.727896e+08,1.826227e+08,1.743744e+08,1.238077e+08,2.219862e+08,2.259524e+08,2.652615e+08,2.203567e+08,1.803391e+08
2,Fruits,6.059709e+05,1.115279e+06,6.677719e+05,6.169646e+05,6.191567e+05,6.361326e+05,5.575049e+05,1.296950e+06,1.225384e+06,1.242626e+06,8.106828e+05,7.755788e+05,5.231983e+05,1.003307e+06,1.029209e+06,1.229794e+06,1.001668e+06,8.056613e+05
3,Meat and Prep,2.028191e+06,3.746174e+06,2.237460e+06,2.066054e+06,2.073391e+06,2.129927e+06,1.864788e+06,4.354019e+06,4.114322e+06,4.169128e+06,2.717240e+06,2.600127e+06,1.749102e+06,3.364792e+06,3.452714e+06,4.129799e+06,3.359856e+06,2.701298e+06
4,Processed Cereals,7.915969e+07,1.323180e+08,8.478277e+07,7.953099e+07,7.979158e+07,8.232285e+07,7.402857e+07,1.566240e+08,1.486267e+08,1.530829e+08,1.043683e+08,9.687001e+07,7.041911e+07,1.252897e+08,1.282419e+08,1.481043e+08,1.234660e+08,9.987445e+07
5,Sugar and Honey,1.563844e+09,2.712624e+09,1.692975e+09,1.579038e+09,1.584345e+09,1.632077e+09,1.453637e+09,3.189108e+09,3.022541e+09,3.093440e+09,2.075967e+09,1.946307e+09,1.375907e+09,2.518903e+09,2.581336e+09,3.019750e+09,2.493557e+09,2.011953e+09
6,Unprocessed Cereals,6.119324e+09,1.130241e+10,6.750663e+09,6.233538e+09,6.255676e+09,6.426260e+09,5.626345e+09,1.313637e+10,1.241321e+10,1.257861e+10,8.198245e+09,7.844833e+09,5.277324e+09,1.015190e+10,1.041716e+10,1.245988e+10,1.013697e+10,8.150058e+09
7,Vegetables,2.204734e+08,4.071367e+08,2.432057e+08,2.245822e+08,2.253798e+08,2.315274e+08,2.027188e+08,4.732139e+08,4.471586e+08,4.531358e+08,2.953502e+08,2.826170e+08,1.901490e+08,3.657231e+08,3.752724e+08,4.488362e+08,3.651829e+08,2.936115e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.95,0.12,0.02,0.02,0.06,1.38,1.25,1.28,0.47,0.41,0.89,0.94,1.32,0.89,0.53
1,Fats and Oils,0.89,0.10,0.01,0.01,0.05,1.29,1.19,1.25,0.46,0.39,0.87,0.90,1.25,0.85,0.50
2,Fruits,0.75,0.09,0.02,0.02,0.04,1.09,0.99,1.01,0.37,0.32,0.71,0.75,1.04,0.71,0.42
3,Meat and Prep,0.72,0.09,0.02,0.02,0.04,1.04,0.94,0.96,0.36,0.31,0.67,0.71,0.99,0.67,0.40
4,Processed Cereals,0.78,0.08,0.01,0.01,0.05,1.22,1.10,1.17,0.45,0.34,0.81,0.85,1.15,0.78,0.43
5,Sugar and Honey,1.82,0.20,0.02,0.03,0.11,2.75,2.48,2.60,0.99,0.78,1.81,1.91,2.60,1.77,1.01
6,Unprocessed Cereals,3.02,0.37,0.07,0.08,0.18,4.38,3.96,4.05,1.50,1.29,2.84,3.00,4.19,2.83,1.68
7,Vegetables,1.92,0.23,0.04,0.05,0.11,2.79,2.52,2.58,0.95,0.82,1.81,1.91,2.67,1.80,1.07
